In [3]:
!pip install ipywidgets

   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   ----------------- ---------------------- 61.4/139.8 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 139.8/139.8 kB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/216.6 kB ? eta -:--:--
   --------------------------------------- 216.6/216.6 kB 13.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------- -------------------------------- 0.4/2.2 MB 13.5 MB/s eta 0:00:01
   ---------- ----------------------------- 0.6/2.2 MB 7.5 MB/s eta 0:00:01
   ------------- -------------------------- 0.7/2.2 MB 5.9 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.2 MB 6.0 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.2 MB 6.0 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.2 MB 3.4 MB/s eta 0:00:01
   ------------------------ --------------- 1.4/2.2 MB 5.1 MB/s eta 0:00:01
   --------------


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\Asus\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Ruthvik\Gov_Support_RAG_Chatbot\ministry_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

In [ ]:
# function to load data files/ pdfs 
def pdf_file_loader(file):
    pdf_loader = DirectoryLoader(file, glob=["*.pdf"], loader_cls=PyPDFLoader) 
    
    pdf_docs = pdf_loader.load()

    return pdf_docs


In [11]:
data = pdf_file_loader(file= 'dataset/')

In [ ]:
data

In [13]:
# Text chunking

def text_chunk(data):
    text_chunker = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    chunks = text_chunker.split_documents(data)

    return chunks

In [16]:
String_bits = text_chunk(data)

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings 

In [28]:
def HGF_embedder(model: str= "sentence-transformers/all-MiniLM-L6-v2"):
    embedding_model = HuggingFaceEmbeddings(model_name= model)

    return embedding_model

In [29]:
text_embedding_model = HGF_embedder()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
#from langchain.vectorstores.cassandra import Cassandra

In [33]:
ASTRA_DB_TOKEN = os.environ.get("ASTRA_DB_TOKEN")
ASTRA_DB_API = os.environ.get("ASTRA_DB_API")

In [ ]:
# # --- 4. Create Embeddings and Vector Store ---
# # GoogleGenerativeAIEmbeddings is the class for Gemini embeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # Recommended embedding model for Gemini

# # Create a FAISS vector store (in-memory for this example)
# vector_store = FAISS.from_documents(docs, embeddings)
# print("Vector store created and populated.")


None


In [ ]:
from langchain_astradb import AstraDBVectorStore

vstore = AstraDBVectorStore(
    collection_name="DocStore",
    embedding = text_embedding_model,
    token= "",
    api_endpoint= ""
)

In [ ]:
vstore.add_documents(String_bits)

### Searching on the vector db

In [ ]:
!pip install "astrapy>=2.0,<3.0"

In [ ]:
from astrapy import DataAPIClient

# Get an existing collection
client = DataAPIClient()
database = client.get_database(
    "API_ENDPOINT",
    token="APPLICATION_TOKEN",
)
collection = database.get_collection("DocStore")

# Find documents
search_query = "Explain the constitution of India"

# Perform the semantic search using the built-in vectorize
search_results = collection.find(
    sort={"$vectorize": search_query},
    limit=3,
    include_similarity=True
)

# Print the results
for doc in results["data"]["documents"]:
    print(doc.get('title', 'Untitled Document'))

ModuleNotFoundError: No module named 'astrapy.db'

### Prompting through LLM

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
     """ You are an assistant for question-answering tasks.
     "Use the following places of retrieved content to answer the question.
     if you don't know the answer, say that you are not aware of that.
     Use three sentence maximum and keep the answer concise."""
     "\n\n"
     "{context}"
)

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm=, prompt=)
rag_chain = create_retrieval_chain(search_results, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input" : "What is Acticle 73 about in constitution of India?"})
print(response["answer"])

In [ ]:
import os
from dotenv import load_dotenv

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate

# Load environment variables (like GOOGLE_API_KEY)
load_dotenv()

# --- 1. Set up Google API Key ---
# LangChain will automatically look for GOOGLE_API_KEY environment variable.
# If not set, you can pass it directly:
# os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"


# # For our dummy text file:
# from langchain_community.document_loaders import TextLoader
# loader = TextLoader("sample.txt")
# documents = loader.load()




# --- 5. Initialize the Gemini Chat Model ---
# ChatGoogleGenerativeAI is the class for Gemini chat models in LangChain
# model="gemini-pro" is the text-only Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.0) # temperature=0.0 for deterministic answers

print(f"Initialized Gemini model: {llm.model_name}")

# --- 6. Set up the QA Chain ---
# We'll use a standard QA chain for RAG
# Prompt template for better answer formatting
prompt_template = """
Answer the question based on the provided context only.
If the answer is not found in the context, politely state that you don't have enough information.

Context:
{context}

Question:
{question}

Answer:
"""
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

# Load the QA chain, specifying the Gemini LLM and prompt
chain = load_qa_chain(llm, chain_type="stuff", prompt=PROMPT) # 'stuff' combines all docs into one prompt

# --- 7. Perform QA ---
question = "What are large language models used for and what role does Gemini play?"
print(f"\n--- Asking Question ---")
print(f"Question: {question}")

# Retrieve relevant documents from the vector store
retrieved_docs = vector_store.similarity_search(question)
print(f"Retrieved {len(retrieved_docs)} relevant document(s).")

# Run the QA chain
try:
    response = chain.run(input_documents=retrieved_docs, question=question)
    print("\n--- Answer ---")
    print(response)
except Exception as e:
    print(f"An error occurred during QA: {e}")
    print("Please ensure your GOOGLE_API_KEY is correct and the Gemini model is accessible.")


# --- Another Question ---
question_2 = "What is the capital of France according to the document?"
print(f"\n--- Asking Another Question ---")
print(f"Question: {question_2}")

retrieved_docs_2 = vector_store.similarity_search(question_2)
print(f"Retrieved {len(retrieved_docs_2)} relevant document(s).")

try:
    response_2 = chain.run(input_documents=retrieved_docs_2, question=question_2)
    print("\n--- Answer ---")
    print(response_2)
except Exception as e:
    print(f"An error occurred during QA: {e}")